# Football Player Positioning AI - LSTM Training

Train a per-player LSTM model on Google Colab with free GPU.

**Important**: Each game's player is a DIFFERENT person. `game1_home_11` != `game2_home_11`.

## 0. Configuration

Set which game and player to train below.

In [ ]:
# ============================================
#  CONFIGURE HERE: which game and player
# ============================================
GAME = 'game1'       # game1, game2, game3
PLAYER = 'home_14'   # e.g. home_14, away_9
DATASET_ID = f'{GAME}_{PLAYER}'
# ============================================

print(f'[CONFIG] Game:   {GAME}')
print(f'[CONFIG] Player: {PLAYER}')
print(f'[CONFIG] ID:     {DATASET_ID}')

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > GPU')

## 2. Clone Repository

In [ ]:
import os
if os.path.exists('football-positioning-ai'):
    print('[DEBUG] Repo exists, pulling latest...')
    !cd football-positioning-ai && git pull
else:
    !git clone https://github.com/Sysus611/football-positioning-ai.git
%cd football-positioning-ai
!pip install pandas numpy pyarrow scipy pyyaml matplotlib -q
print('[OK] Ready')

## 3. Download Data (only the selected game)

In [ ]:
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data"
FILES = {
    'game1': {
        'tracking_home.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv',
    },
    'game2': {
        'tracking_home.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Away_Team.csv',
    },
    'game3': {
        'tracking.txt': 'Sample_Game_3/Sample_Game_3_tracking.txt',
        'metadata.xml': 'Sample_Game_3/Sample_Game_3_metadata.xml',
        'metadata.json': 'Sample_Game_3/Sample_Game_3_events.json',
    },
}

file_map = FILES[GAME]
game_dir = f'data/raw/metrica/{GAME}'
os.makedirs(game_dir, exist_ok=True)

print(f'[DEBUG] Downloading {GAME} data...')
for local_name, remote_path in file_map.items():
    dest = f'{game_dir}/{local_name}'
    if os.path.exists(dest):
        print(f'  [SKIP] {local_name} already exists')
        continue
    print(f'  [DOWNLOAD] {remote_path}...', end='')
    urllib.request.urlretrieve(f'{BASE_URL}/{remote_path}', dest)
    print(f' OK ({os.path.getsize(dest)/1024/1024:.1f} MB)')
print('[OK] Download complete')

## 4. Preprocessing (selected game only)

In [ ]:
import time
t0 = time.time()
!python src/preprocessing/preprocess.py
print(f'\n[OK] Preprocessing done in {time.time()-t0:.0f}s')

## 5. Feature Engineering (selected game + player only)

In [ ]:
import time
t0 = time.time()
!python src/features/build_features.py {GAME} {PLAYER}
print(f'\n[OK] Feature engineering done in {time.time()-t0:.0f}s')

## 6. Train

In [ ]:
import time
print(f'[DEBUG] Training {DATASET_ID}...')
t0 = time.time()
!python src/training/train.py {DATASET_ID}
print(f'\n[OK] Training done in {time.time()-t0:.0f}s ({(time.time()-t0)/60:.1f} min)')

## 7. Results

In [ ]:
import os, torch
ckpt = torch.load(f'data/models/{DATASET_ID}.pt', map_location='cpu', weights_only=False)
val_loss = ckpt['best_val_loss']
dist_m = (val_loss ** 0.5) * ((105 + 68) / 2)
print(f'[RESULT] {DATASET_ID}')
print(f'  Val Loss: {val_loss:.6f}')
print(f'  ~Distance Error: {dist_m:.2f}m')
print(f'  Best Epoch: {ckpt["epoch"]}')

## 8. Visualization: Prediction vs Actual (Real Pitch)

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt, sys
from matplotlib.patches import Arc, Circle, Rectangle
sys.path.insert(0, '.')
from src.model.lstm_baseline import PlayerLSTM

# --- Draw real-scale football pitch ---
def draw_pitch(ax, pitch_length=105, pitch_width=68):
    ax.set_facecolor('#1a472a')
    ax.set_xlim(-3, pitch_length + 3)
    ax.set_ylim(-3, pitch_width + 3)
    ax.set_aspect('equal')
    ax.tick_params(colors='#aaa', labelsize=7)
    lw, c = 1.2, 'white'
    # Boundary
    ax.plot([0,0],[0,pitch_width], c, lw=lw)
    ax.plot([0,pitch_length],[pitch_width,pitch_width], c, lw=lw)
    ax.plot([pitch_length,pitch_length],[pitch_width,0], c, lw=lw)
    ax.plot([pitch_length,0],[0,0], c, lw=lw)
    # Halfway line + center circle
    ax.plot([pitch_length/2, pitch_length/2], [0, pitch_width], c, lw=lw)
    ax.add_patch(Circle((pitch_length/2, pitch_width/2), 9.15, fill=False, ec=c, lw=lw))
    ax.scatter(pitch_length/2, pitch_width/2, c=c, s=15, zorder=5)
    # Left penalty area
    hw = pitch_width / 2
    ax.plot([0,16.5],[hw+20.16, hw+20.16], c, lw=lw)
    ax.plot([16.5,16.5],[hw+20.16, hw-20.16], c, lw=lw)
    ax.plot([16.5,0],[hw-20.16, hw-20.16], c, lw=lw)
    ax.plot([0,5.5],[hw+9.16, hw+9.16], c, lw=lw)
    ax.plot([5.5,5.5],[hw+9.16, hw-9.16], c, lw=lw)
    ax.plot([5.5,0],[hw-9.16, hw-9.16], c, lw=lw)
    ax.scatter(11, hw, c=c, s=12, zorder=5)
    ax.add_patch(Arc((11, hw), 18.3, 18.3, angle=0, theta1=-53.13, theta2=53.13, ec=c, lw=lw))
    # Right penalty area
    L = pitch_length
    ax.plot([L,L-16.5],[hw+20.16, hw+20.16], c, lw=lw)
    ax.plot([L-16.5,L-16.5],[hw+20.16, hw-20.16], c, lw=lw)
    ax.plot([L-16.5,L],[hw-20.16, hw-20.16], c, lw=lw)
    ax.plot([L,L-5.5],[hw+9.16, hw+9.16], c, lw=lw)
    ax.plot([L-5.5,L-5.5],[hw+9.16, hw-9.16], c, lw=lw)
    ax.plot([L-5.5,L],[hw-9.16, hw-9.16], c, lw=lw)
    ax.scatter(L-11, hw, c=c, s=12, zorder=5)
    ax.add_patch(Arc((L-11, hw), 18.3, 18.3, angle=0, theta1=126.87, theta2=233.13, ec=c, lw=lw))

# --- Load model ---
ckpt = torch.load(f'data/models/{DATASET_ID}.pt', map_location='cpu', weights_only=False)
model = PlayerLSTM(input_dim=ckpt['input_dim'], pred_frames=ckpt['pred_frames'])
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

data = np.load(f'data/tensors/{DATASET_ID}.npz')
X_val, Y_val = data['X_val'], data['Y_val']
print(f'Validation samples: {len(X_val)}')

PITCH_L, PITCH_W = 105, 68

# --- 3x3 = 9 sample plots ---
fig, axes = plt.subplots(3, 3, figsize=(24, 16))
fig.suptitle(f'{DATASET_ID} — Prediction vs Ground Truth', fontsize=18, color='white', fontweight='bold')
fig.patch.set_facecolor('#111')

for ax in axes.flat:
    idx = np.random.randint(len(X_val))
    x_sample = X_val[idx]       # [75, 48]
    last_frame = x_sample[-1]   # last obs frame features

    with torch.no_grad():
        pred = model(torch.from_numpy(X_val[idx:idx+1])).numpy()[0]  # [50,2]
    actual = Y_val[idx]  # [50,2]

    draw_pitch(ax)

    # Other players: features[0:42] = 21 x (x,y)
    other_coords = last_frame[0:42].reshape(21, 2)
    for i in range(21):
        px, py = other_coords[i, 0] * PITCH_L, other_coords[i, 1] * PITCH_W
        if px == 0 and py == 0:
            continue
        color = '#4da6ff' if i < 10 else '#ff6b6b'
        ax.scatter(px, py, c=color, s=60, edgecolors='white', linewidths=0.5, zorder=6, alpha=0.8)

    # Ball: features[42:44]
    ball_x, ball_y = last_frame[42] * PITCH_L, last_frame[43] * PITCH_W
    ax.scatter(ball_x, ball_y, c='yellow', s=100, edgecolors='black', linewidths=1, zorder=8, marker='o', label='Ball')

    # Target player past trajectory: features[44:46]
    obs_x = x_sample[:, 44] * PITCH_L
    obs_y = x_sample[:, 45] * PITCH_W
    ax.plot(obs_x, obs_y, color='cyan', lw=2, alpha=0.5, label='Past 3s')
    ax.scatter(obs_x[0], obs_y[0], c='cyan', s=40, marker='s', zorder=7)
    ax.scatter(obs_x[-1], obs_y[-1], c='cyan', s=80, marker='o', edgecolors='white', linewidths=1.5, zorder=9)

    # Actual future
    act_x, act_y = actual[:, 0] * PITCH_L, actual[:, 1] * PITCH_W
    ax.plot(act_x, act_y, 'w-', lw=2.5, alpha=0.9, label='Actual future')
    ax.scatter(act_x[-1], act_y[-1], c='white', s=100, marker='*', zorder=9)

    # Predicted future
    pred_x, pred_y = pred[:, 0] * PITCH_L, pred[:, 1] * PITCH_W
    ax.plot(pred_x, pred_y, color='#ff4444', ls='--', lw=2.5, alpha=0.9, label='Predicted')
    ax.scatter(pred_x[-1], pred_y[-1], c='#ff4444', s=100, marker='*', zorder=9)

    # Error metrics
    err = np.sqrt(((pred - actual)**2).sum(axis=1)).mean() * ((PITCH_L + PITCH_W) / 2)
    end_err = np.sqrt(((pred[-1] - actual[-1])**2).sum()) * ((PITCH_L + PITCH_W) / 2)
    ax.set_title(f'#{idx}  avg:{err:.1f}m  end:{end_err:.1f}m', color='white', fontsize=11, pad=4)
    ax.legend(fontsize=7, loc='upper right', framealpha=0.6)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('prediction_vis.png', dpi=150, bbox_inches='tight', facecolor='#111')
plt.show()
print('[OK] Saved prediction_vis.png')

## 9. Download Model

In [ ]:
from google.colab import files
files.download(f'data/models/{DATASET_ID}.pt')